In [1]:
import sqlite3
import pandas as pd 

conn= sqlite3.connect('flightdelay_project.db')

### Create the Merged Flight & Weather Table

 `INNER JOIN` combines each flight with the daily weather recorded at its origin airport. Flights without matching weather data are excluded, leaving records from only the 20 selected airports.

In [2]:
create_merged= """
DROP TABLE IF EXISTS merged_flight_weather;

CREATE TABLE merged_flight_weather AS 
SELECT
   flights_cleaned.FLIGHT_DATE, 
   flights_cleaned.MONTH, 
   flights_cleaned.DAY_OF_MONTH,
   flights_cleaned.DAY_OF_WEEK,
   flights_cleaned.ORIGIN,
   flights_cleaned.DESTINATION,
   flights_cleaned.ROUTE,
   flights_cleaned.AIRLINE,
   flights_cleaned.SCHEDULED_DEP_TIME,
   flights_cleaned.DEPARTURE_HOUR,
   flights_cleaned.DISTANCE, 
   flights_cleaned.IS_DELAYED,

   weather_cleaned.WEATHER_DATE,
   weather_cleaned.AIRPORT, 
   weather_cleaned.TEMPERATURE,
   weather_cleaned.VISIBILITY,
   weather_cleaned.WINDSPEED,
   weather_cleaned.PRECIPITATION,
   weather_cleaned.WEATHER_FLAGS
FROM flights_cleaned
INNER JOIN weather_cleaned
 ON flights_cleaned.FLIGHT_DATE= weather_cleaned.WEATHER_DATE
 AND flights_cleaned.ORIGIN= weather_cleaned.AIRPORT;
"""

conn.executescript (create_merged)
conn.commit() 



In [3]:
merged_preview= """
SELECT * FROM merged_flight_weather LIMIT 10; 
"""
pd.read_sql_query(merged_preview, conn)

,FLIGHT_DATE,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DESTINATION,ROUTE,AIRLINE,SCHEDULED_DEP_TIME,DEPARTURE_HOUR,DISTANCE,IS_DELAYED,WEATHER_DATE,AIRPORT,TEMPERATURE,VISIBILITY,WINDSPEED,PRECIPITATION,WEATHER_FLAGS
0,2025-01-01,1,1,3,SFO,JFK,SFO-JFK,AA,1030,10,2586.0,0,2025-01-01,SFO,49.8,10.0,4.4,0.00,000000
1,2025-01-01,1,1,3,BOS,LAX,BOS-LAX,AA,801,8,2611.0,0,2025-01-01,BOS,44.4,6.9,8.7,0.46,010000
2,2025-01-01,1,1,3,LAX,JFK,LAX-JFK,AA,1130,11,2475.0,0,2025-01-01,LAX,53.3,3.6,4.7,0.00,100000
3,2025-01-01,1,1,3,PHX,OMA,PHX-OMA,AA,2025,20,1037.0,0,2025-01-01,PHX,58.0,8.5,2.8,0.00,000000
4,2025-01-01,1,1,3,PHX,SMF,PHX-SMF,AA,1104,11,647.0,1,2025-01-01,PHX,58.0,8.5,2.8,0.00,000000
5,2025-01-01,1,1,3,DFW,CLE,DFW-CLE,AA,1322,13,1021.0,0,2025-01-01,DFW,44.7,10.0,6.8,0.00,000000
6,2025-01-01,1,1,3,DFW,MEM,DFW-MEM,AA,819,8,431.0,0,2025-01-01,DFW,44.7,10.0,6.8,0.00,000000
7,2025-01-01,1,1,3,CLT,BNA,CLT-BNA,AA,1651,16,328.0,0,2025-01-01,CLT,50.6,10.0,8.2,0.00,000000
8,2025-01-01,1,1,3,CLT,EWR,CLT-EWR,AA,1100,11,529.0,0,2025-01-01,CLT,50.6,10.0,8.2,0.00,000000
9,2025-01-01,1,1,3,CLT,SAV,CLT-SAV,AA,2248,22,213.0,0,2025-01-01,CLT,50.6,10.0,8.2,0.00,000000


In [4]:
merged_datacheck= """
SELECT 
 COUNT(*) AS ROW_COUNT, 
 COUNT (DISTINCT AIRPORT) AS AIRPORT_COUNT,
 MIN (FLIGHT_DATE) AS START_DATE,
 MAX (FLIGHT_DATE) AS END_DATE,
    SUM(CASE WHEN FLIGHT_DATE IS NULL THEN 1 ELSE 0 END) AS FLIGHT_DATE_MISSING,
    SUM(CASE WHEN ORIGIN IS NULL THEN 1 ELSE 0 END) AS ORIGIN_MISSING,
    SUM(CASE WHEN IS_DELAYED IS NULL THEN 1 ELSE 0 END) AS DELAY_STATUS_MISSING,
    SUM(CASE WHEN TEMPERATURE IS NULL THEN 1 ELSE 0 END) AS TEMPERATURE_MISSING,
    SUM(CASE WHEN VISIBILITY IS NULL THEN 1 ELSE 0 END) AS VISIBILITY_MISSING,
    SUM(CASE WHEN WINDSPEED IS NULL THEN 1 ELSE 0 END) AS WINDSPEED_MISSING,
    SUM(CASE WHEN PRECIPITATION IS NULL THEN 1 ELSE 0 END) AS PRECIPITATION_MISSING,
    SUM(CASE WHEN WEATHER_FLAGS IS NULL THEN 1 ELSE 0 END) AS WEATHER_FLAGS_MISSING
FROM merged_flight_weather; 
"""
pd.read_sql_query(merged_datacheck, conn)

,ROW_COUNT,AIRPORT_COUNT,START_DATE,END_DATE,FLIGHT_DATE_MISSING,ORIGIN_MISSING,DELAY_STATUS_MISSING,TEMPERATURE_MISSING,VISIBILITY_MISSING,WINDSPEED_MISSING,PRECIPITATION_MISSING,WEATHER_FLAGS_MISSING
0,1141948,20,2025-01-01,2025-04-30,0,0,0,0,0,0,0,0


### Merge Validation

The final merged dataset contains 1,141,948 flights from all 20 selected airports with no missing model values. `INNER JOIN` has excluded 1,046,828 flights from airports without matching weather data; these rows were intentionally removed to prepare for the small-scale machine learning predictor model.

## SQL Exploratory Data Analysis

Following SQL queries will be used to spot-check delay patterns & confirm that the merged dataset is ready for more detailed Python analysis and modeling.

### Delay-Rate by Airline

In [5]:
delayrate_airline= """ 
SELECT 
 AIRLINE,
 COUNT(*) AS TOTAL_FLIGHTS,
 SUM(IS_DELAYED) AS TOTAL_DELAYS,
 ROUND(100.0 * SUM(IS_DELAYED) / COUNT(*), 2) AS DELAY_RATE_PERCENT
FROM merged_flight_weather
GROUP BY AIRLINE
ORDER BY DELAY_RATE_PERCENT DESC; 
"""
pd.read_sql_query (delayrate_airline, conn) 

,AIRLINE,TOTAL_FLIGHTS,TOTAL_DELAYS,DELAY_RATE_PERCENT
0,OH,37407,11032,29.49
1,F9,39451,9881,25.05
2,B6,32898,7730,23.50
3,AA,202761,46425,22.90
4,AS,37448,7875,21.03
5,OO,130058,26804,20.61
6,DL,207431,41680,20.09
7,MQ,45721,9154,20.02
8,WN,136833,26574,19.42
9,HA,2123,392,18.46



- Among airlines with at least 30,000 flights within this dataset, PSA (`OH`) had the highest delay rate at 29.49%, while Republic (`YX`) had the lowest at 17.85%.
- Hawaiian (`HA`) and Allegiant (`G4`) had much smaller sample sizes, so their rates are not directly comparable.

*The 20 airports were selected because they had the highest origin-flight counts in the BTS dataset.
*Source: BTS Carrier Reference Codes.

### Delay-Rate by Origin

In [6]:
delayrate_origin= """ 
SELECT 
 ORIGIN,
 COUNT(*) AS TOTAL_FLIGHTS, 
 SUM(IS_DELAYED) AS DELAYED_FLIGHTS,
 ROUND(100.0 * SUM(IS_DELAYED) / COUNT(*), 2) AS DELAY_RATE_PERCENT
FROM merged_flight_weather
GROUP BY ORIGIN
ORDER BY DELAY_RATE_PERCENT DESC; 
"""
pd.read_sql_query (delayrate_origin, conn) 

,ORIGIN,TOTAL_FLIGHTS,DELAYED_FLIGHTS,DELAY_RATE_PERCENT
0,DCA,44704,11366,25.43
1,DFW,96901,24444,25.23
2,MIA,39787,9306,23.39
3,DEN,98423,22752,23.12
4,CLT,65854,15033,22.83
5,ATL,96518,21759,22.54
6,ORD,90910,20120,22.13
7,BOS,44964,9818,21.84
8,EWR,39034,8426,21.59
9,DTW,37808,8140,21.53


- Highest delay-rate for airports within dataset is Ronald Reagan Washington National Airport `DCA` at 25.43%. 
- Lowest recorded delay-rate is 14.53% at Los Angeles International Airport `LAX`. 

In [7]:
delayrate_month= """ 
SELECT
 MONTH,
 COUNT(*) AS TOTAL_FLIGHTS,
 SUM(IS_DELAYED) AS TOTAL_DELAYS,
 ROUND(100.0 * SUM(IS_DELAYED) / COUNT(*),2) AS DELAY_RATE_PERCENT
FROM merged_flight_weather
GROUP BY MONTH
ORDER BY DELAY_RATE_PERCENT DESC;
"""

pd.read_sql_query (delayrate_month, conn) 

,MONTH,TOTAL_FLIGHTS,TOTAL_DELAYS,DELAY_RATE_PERCENT
0,2,260148,56878,21.86
1,4,299050,62578,20.93
2,3,307465,63711,20.72
3,1,275285,53337,19.38


- Monthly delay rates are fairly similar, ranging from 19.38% to 21.86%. February recorded the highest delay rate at 21.86%, while January recorded the lowest at 19.38%.

### Delay-Rate on Dry vs. Precipitation Days
This analysis compares flight delay rates on days with and without measurable precipitation.

In [12]:
delayrate_precip= """ 
SELECT
 CASE 
    WHEN PRECIPITATION = 0 THEN 'No measurable precipitation'
    ELSE 'Measurable precipitation'
    END AS PRECIPITATION_GROUP,
 COUNT (*) AS TOTAL_FLIGHTS,
 SUM (IS_DELAYED) AS DELAYED_FLIGHTS,
 ROUND(100.0 * SUM(IS_DELAYED) / COUNT(*), 2) AS DELAY_RATE_PERCENT
FROM merged_flight_weather
GROUP BY PRECIPITATION_GROUP
ORDER BY DELAY_RATE_PERCENT DESC; 
"""

pd.read_sql_query (delayrate_precip, conn) 

,PRECIPITATION_GROUP,TOTAL_FLIGHTS,DELAYED_FLIGHTS,DELAY_RATE_PERCENT
0,Measurable precipitation,317262,86812,27.36
1,No measurable precipitation,824686,149692,18.15


- Delay-rate reports at 27.36% on days with measurable precipitation compared to 18.15% delay-rate on days with no measured precipitation.

In [15]:
#load merged table into Pandas dataframe:
merged_df= pd.read_sql_query("SELECT * FROM merged_flight_weather;", conn)

#save merged dataset as CSV:
merged_df.to_csv(r"C:\Users\what1\OneDrive\Documents\Flight Delay Prediction Model\data\merged\merged_flight_weather.csv",
    index=False
)


print("Merged CSV saved.")


Merged CSV saved.
